# Set Up Environment

## Import Libraries

In [27]:
import numpy as np
import pandas as pd
import geopandas as gpd
# import pygwalker as pyg

import io
import os
import requests

from dotenv import load_dotenv

load_dotenv()


True

## Set Environment Variables

In [28]:
PROJECT_CRS = "EPSG:3566"


## Helper Functions

In [29]:
def fetch_github(
    url: str, mode: str = "private", token_env_var: str = "GITHUB_TOKEN"
) -> requests.Response:
    """
    Fetch content from GitHub repositories.

    Args:
        url: GitHub raw URL (e.g., https://raw.githubusercontent.com/...)
        mode: "public" for public repos, "private" for private repos requiring authentication
        token_env_var: Name of environment variable containing GitHub token (default: GITHUB_TOKEN)

    Returns:
        requests.Response object

    Raises:
        ValueError: If token is missing for private mode or invalid mode
        requests.HTTPError: If request fails
    """

    # Validate mode
    if mode not in ["public", "private"]:
        raise ValueError(f"mode must be 'public' or 'private', got '{mode}'")

    if mode == "public":
        response = requests.get(url, timeout=30)
    else:
        token = os.getenv(token_env_var)
        if not token:
            raise ValueError(
                f"GitHub token not found in environment variable '{token_env_var}'. "
                f"Check your .env file has: {token_env_var}=your_token_here"
            )

        headers = {
            "Authorization": f"token {token}",
            "Accept": "application/vnd.github.v3.raw",
        }
        response = requests.get(url, headers=headers, timeout=30)

    response.raise_for_status()
    return response


In [30]:
# Create extrapolate function analogous to pandas interpolate
def extrapolate(series, x_values):
    """Extrapolate missing values using linear regression on available data"""
    mask = series.notna()
    if mask.sum() < 2:  # Need at least 2 points for regression
        return series.ffill()

    from scipy import stats

    slope, intercept, _, _, _ = stats.linregress(x_values[mask], series[mask])
    return series.where(series.notna(), slope * x_values + intercept)


# Input Data

## Preprocessed Forecast Results

In [31]:
forecast_results = pd.read_csv("results/final_forecast_df.csv")
forecast_results[["externalid", "year", "final_forecast"]]


,externalid,year,final_forecast
0,3601,2027,900
1,3601,2032,1000
2,3601,2036,1100
3,3601,2046,1300
4,3601,2055,1400
...,...,...,...
169,3629,2032,4200
170,3629,2036,4600
171,3629,2046,5400
172,3629,2055,6200


## Traffic Factors

In [32]:
gdf_master_segments = gpd.read_file(
    "zip://data/updated-traffic-factors/Master_Segs_withFactors_20251120.zip"
).to_crs(PROJECT_CRS)

gdf_master_segments


,SEGID,BMP,EMP,DISTANCE,CO_FIPS,PLANAREA,AADT2023,AADT2022,AADT2021,AADT2020,...,FAC_WIN,FAC_SPR,FAC_SUM,FAC_FAL,FAC_MAXMO,FAC_MAX,FACMANADJ,SUTRUCKS,CUTRUCKS,geometry
0,0006_000.0,0.000,0.665,0.666641,27,UDOT,457.0,441.0,474.0,430.0,...,0.8769,1.0071,1.0496,1.0664,10,1.1275,0,0.2496,0.2324,"LINESTRING (916692.404 6835614.744, 920182.189..."
1,0006_000.7,0.665,16.022,15.369839,27,UDOT,457.0,441.0,474.0,430.0,...,0.8769,1.0071,1.0496,1.0664,10,1.1275,0,0.2496,0.2324,"LINESTRING (920182.189 6835168.248, 923166.425..."
2,0006_016.0,16.022,46.017,30.001961,27,UDOT,457.0,441.0,474.0,430.0,...,0.8769,1.0071,1.0496,1.0664,10,1.1275,0,0.2496,0.2324,"LINESTRING (999430.463 6834408.584, 999447.155..."
3,0006_046.0,46.017,60.218,14.194306,27,UDOT,409.0,395.0,424.0,385.0,...,0.8769,1.0071,1.0496,1.0664,10,1.1275,0,0.1751,0.3338,"LINESTRING (1143701.911 6830874.803, 1145024.9..."
4,0006_060.2,60.218,77.545,17.323237,27,UDOT,409.0,395.0,424.0,385.0,...,0.8769,1.0071,1.0496,1.0664,10,1.1275,0,0.1751,0.3338,"LINESTRING (1206604.946 6871212.763, 1206701.0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9252,WFRC_8489,0.000,0.000,0.505514,35,WFRC,0.0,0.0,0.0,0.0,...,0.9411,0.9924,1.0246,1.0419,8,1.0510,0,0.1086,0.0565,"LINESTRING (1518588.41 7388436.516, 1518656.54..."
9253,WFRC_8490,0.000,0.000,0.737896,35,WFRC,0.0,0.0,0.0,0.0,...,0.9411,0.9924,1.0246,1.0419,8,1.0510,0,0.1086,0.0565,"LINESTRING (1521238.95 7388420.585, 1522418.15..."
9254,WFRC_8491,0.000,0.000,0.265495,35,WFRC,0.0,0.0,0.0,0.0,...,0.9411,0.9924,1.0246,1.0419,8,1.0510,0,0.1086,0.0565,"LINESTRING (1525103.396 7388831.353, 1525935.1..."
9255,WFRC_8492,0.000,0.000,0.444725,35,WFRC,0.0,0.0,0.0,0.0,...,0.9519,1.0137,1.0141,1.0203,5,1.0436,0,0.1052,0.0432,"LINESTRING (1527315.01 7447662.776, 1527326.78..."


## AADT and Truck Percentages

In [33]:
# Read Processed UDOT AADT Daya directly from GitHub repo
response = fetch_github(
    "https://raw.githubusercontent.com/WFRCAnalytics/DATA-UDOT-AADT-Processing/refs/heads/main/_output/udot_aadt_trkpct_data.csv",
    mode="private",
)

df_aadt_udot = pd.read_csv(io.StringIO(response.text))

df_aadt_udot


,Station,RouteID,BeginPoint,EndPoint,SectionLength,DESC_,YEAR,AADT,SUTRK,CUTRK
0,001-0010,0015PM,109.029,112.071,3.042,SR 160 South Beaver Milford,1981,4100,0.029041,0.222085
1,001-0010,0015PM,109.029,112.071,3.042,SR 160 South Beaver Milford,1982,4300,0.029041,0.222085
2,001-0010,0015PM,109.029,112.071,3.042,SR 160 South Beaver Milford,1983,4600,0.029041,0.222085
3,001-0010,0015PM,109.029,112.071,3.042,SR 160 South Beaver Milford,1984,4800,0.029041,0.222085
4,001-0010,0015PM,109.029,112.071,3.042,SR 160 South Beaver Milford,1985,5100,0.029041,0.222085
...,...,...,...,...,...,...,...,...,...,...
199535,057-1530,3424PM,0.553,1.306,0.753,9th St (Rt 3426) via Polk Ave - Sheridan Dr,2020,2200,0.000000,0.000000
199536,057-1530,3424PM,0.553,1.306,0.753,9th St (Rt 3426) via Polk Ave - Sheridan Dr,2021,2400,0.000000,0.000000
199537,057-1530,3424PM,0.553,1.306,0.753,9th St (Rt 3426) via Polk Ave - Sheridan Dr,2022,2400,0.000000,0.000000
199538,057-1530,3424PM,0.553,1.306,0.753,9th St (Rt 3426) via Polk Ave - Sheridan Dr,2023,2500,0.000000,0.000000


## Load External-Segment Link

In [34]:
external_segment_link = pd.read_csv("params/externals-segments-link.csv")
external_segment_link


,externalid,segid
0,3601,1082_000.0
1,3602,0013_006.5
2,3603,1112_000.0
3,3604,0015_368.1
4,3605,0038_003.2
5,3606,0091_010.1
6,3607,3462_002.8
7,3608,0039_008.7
8,3609,0084_087.8
9,3610,2688_005.5


# Prepare Data

## Load Previous Base Year Data

In [35]:
df_external_year = pd.read_csv(r"archive/v920/external_year_vol.csv")

df_external_year


,;Idx_WF,WF_Ext,Year,AWDT,PASS_VOL,TRUCK_MD,TRUCK_HV,AWDT_FAC,AADT,PASSENGER,TRUCK_SU,TRUCK_MU,PctTrk_SU,PctTrk_MU
0,36012010,3601,2010,492,321,105,66,0.956,515,336,110,69,0.214,0.135
1,36012011,3601,2011,487,317,104,66,0.956,510,332,109,69,0.214,0.135
2,36012012,3601,2012,558,364,119,75,0.956,585,381,125,79,0.214,0.135
3,36012013,3601,2013,577,377,123,77,0.956,605,395,129,81,0.214,0.135
4,36012014,3601,2014,587,382,126,79,0.956,615,400,132,83,0.214,0.135
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1474,36292056,3629,2056,3409,2291,637,481,0.984,3464,2328,647,489,0.187,0.141
1475,36292057,3629,2057,3457,2323,646,488,0.984,3513,2361,656,496,0.187,0.141
1476,36292058,3629,2058,3505,2356,654,495,0.984,3562,2394,665,503,0.187,0.141
1477,36292059,3629,2059,3553,2388,663,502,0.984,3611,2427,674,510,0.187,0.141


## Initialize the Final Dataframe

In [36]:
# Step 1: Initiate dataframe with the id, and year columns
df_external_year_update = (
    pd.MultiIndex.from_product(
        [
            external_segment_link["externalid"].unique(),  # Unique External IDs
            range(1981, 2061),  # Range goes up to, but does not include, 2061
        ],
        names=["WF_Ext", "Year"],
    )
    .to_frame(index=False)
    .reset_index(drop=True)
)

# Generate Idx column
df_external_year_update[";Idx_WF"] = df_external_year_update["WF_Ext"].astype(
    str
) + df_external_year_update["Year"].astype(str)

df_external_year_update


,WF_Ext,Year,;Idx_WF
0,3601,1981,36011981
1,3601,1982,36011982
2,3601,1983,36011983
3,3601,1984,36011984
4,3601,1985,36011985
...,...,...,...
2315,3629,2056,36292056
2316,3629,2057,36292057
2317,3629,2058,36292058
2318,3629,2059,36292059


## Segment ID

In [37]:
# Step 2: Join Segment ID from External-Segment Link
df_external_year_update["segid"] = df_external_year_update["WF_Ext"].map(
    external_segment_link.set_index("externalid")["segid"]
)

df_external_year_update


,WF_Ext,Year,;Idx_WF,segid
0,3601,1981,36011981,1082_000.0
1,3601,1982,36011982,1082_000.0
2,3601,1983,36011983,1082_000.0
3,3601,1984,36011984,1082_000.0
4,3601,1985,36011985,1082_000.0
...,...,...,...,...
2315,3629,2056,36292056,1826_004.9
2316,3629,2057,36292057,1826_004.9
2317,3629,2058,36292058,1826_004.9
2318,3629,2059,36292059,1826_004.9


## RouteID and MilePoint

In [38]:
df_external_year_update = df_external_year_update.assign(
    route=lambda x: x["segid"].str.split("_").str[0] + "PM",
    milepost=lambda x: pd.to_numeric(x["segid"].str.split("_").str[1], errors="coerce"),
)

df_external_year_update


,WF_Ext,Year,;Idx_WF,segid,route,milepost
0,3601,1981,36011981,1082_000.0,1082PM,0.0
1,3601,1982,36011982,1082_000.0,1082PM,0.0
2,3601,1983,36011983,1082_000.0,1082PM,0.0
3,3601,1984,36011984,1082_000.0,1082PM,0.0
4,3601,1985,36011985,1082_000.0,1082PM,0.0
...,...,...,...,...,...,...
2315,3629,2056,36292056,1826_004.9,1826PM,4.9
2316,3629,2057,36292057,1826_004.9,1826PM,4.9
2317,3629,2058,36292058,1826_004.9,1826PM,4.9
2318,3629,2059,36292059,1826_004.9,1826PM,4.9


## External Station Name

In [39]:
df_external_year_update["Ext_Name"] = df_external_year_update["WF_Ext"].map(
    forecast_results[["externalid", "external"]]
    .drop_duplicates()
    .set_index("externalid")[
        "external"  # Name of exrernal station
    ]
)

df_external_year_update


,WF_Ext,Year,;Idx_WF,segid,route,milepost,Ext_Name
0,3601,1981,36011981,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge
1,3601,1982,36011982,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge
2,3601,1983,36011983,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge
3,3601,1984,36011984,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge
4,3601,1985,36011985,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge
...,...,...,...,...,...,...,...
2315,3629,2056,36292056,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms
2316,3629,2057,36292057,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms
2317,3629,2058,36292058,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms
2318,3629,2059,36292059,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms


## Adjustment Factors

In [40]:
# # Step 3: Add Average Weekday Factors
df_external_year_update["AWDT_FAC"] = df_external_year_update["segid"].map(
    gdf_master_segments.set_index("SEGID")["FAC_WDAVG"]  # Average Weekday Factors
)

df_external_year_update


,WF_Ext,Year,;Idx_WF,segid,route,milepost,Ext_Name,AWDT_FAC
0,3601,1981,36011981,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN
1,3601,1982,36011982,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN
2,3601,1983,36011983,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN
3,3601,1984,36011984,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN
4,3601,1985,36011985,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN
...,...,...,...,...,...,...,...,...
2315,3629,2056,36292056,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404
2316,3629,2057,36292057,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404
2317,3629,2058,36292058,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404
2318,3629,2059,36292059,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404


## Historic AADT and Truck Percentages (1981 - 2023)

In [41]:
# # Step 4: Add Historic AADT and Truck Percent Data (1981 - 2023)
df_external_year_update[["AADT_Historic", "PctTrk_SU", "PctTrk_MU"]] = (
    df_external_year_update.reset_index()
    .merge(
        df_aadt_udot[["YEAR", "RouteID", "BeginPoint", "AADT", "SUTRK", "CUTRK"]],
        left_on=["Year", "route"],
        right_on=["YEAR", "RouteID"],
        how="left",
        indicator=True,
    )
    .query('_merge == "both"')
    .assign(distance=lambda x: np.abs(x["BeginPoint"] - x["milepost"]))
    .sort_values("distance")
    .groupby("index", dropna=False)
    .first()[["AADT", "SUTRK", "CUTRK"]]
    .rename(
        columns={"AADT": "AADT_Historic", "SUTRK": "PctTrk_SU", "CUTRK": "PctTrk_MU"}
    )
    .reindex(df_external_year_update.index)
)

df_external_year_update


,WF_Ext,Year,;Idx_WF,segid,route,milepost,Ext_Name,AWDT_FAC,AADT_Historic,PctTrk_SU,PctTrk_MU
0,3601,1981,36011981,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0
1,3601,1982,36011982,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0
2,3601,1983,36011983,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0
3,3601,1984,36011984,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0
4,3601,1985,36011985,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
2315,3629,2056,36292056,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,NaN,NaN
2316,3629,2057,36292057,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,NaN,NaN
2317,3629,2058,36292058,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,NaN,NaN
2318,3629,2059,36292059,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,NaN,NaN


# Forecasted AADT (2027 - 2060)

In [42]:
# Step 5: Add Forecasted AADT Data (2027 - 2060)
df_external_year_update["AADT_Forecast"] = (
    df_external_year_update.set_index(["WF_Ext", "Year"]).index.map(
        forecast_results.set_index(["externalid", "year"])["final_forecast"]
    )
).astype("Int64")

df_external_year_update


,WF_Ext,Year,;Idx_WF,segid,route,milepost,Ext_Name,AWDT_FAC,AADT_Historic,PctTrk_SU,PctTrk_MU,AADT_Forecast
0,3601,1981,36011981,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>
1,3601,1982,36011982,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>
2,3601,1983,36011983,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>
3,3601,1984,36011984,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>
4,3601,1985,36011985,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...
2315,3629,2056,36292056,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,NaN,NaN,<NA>
2316,3629,2057,36292057,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,NaN,NaN,<NA>
2317,3629,2058,36292058,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,NaN,NaN,<NA>
2318,3629,2059,36292059,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,NaN,NaN,<NA>


## Combine to AADT

In [43]:
# We combine Historic and Forecasted on the fly, then group and interpolate.
# .combine_first() prioritizes the first column, then fills with the second.
df_external_year_update["AADT"] = (
    df_external_year_update["AADT_Historic"]
    .combine_first(
        df_external_year_update["AADT_Forecast"]
    )  # Get Forecasted AADT and interpolate for others
    .groupby(df_external_year_update["WF_Ext"])
    .transform(
        lambda x: x.interpolate(method="linear")
    )  # Interpolate AADT for 2025 and 2026
    .round()
    .astype("Int64")
)

df_external_year_update


,WF_Ext,Year,;Idx_WF,segid,route,milepost,Ext_Name,AWDT_FAC,AADT_Historic,PctTrk_SU,PctTrk_MU,AADT_Forecast,AADT
0,3601,1981,36011981,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0
1,3601,1982,36011982,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0
2,3601,1983,36011983,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0
3,3601,1984,36011984,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0
4,3601,1985,36011985,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2315,3629,2056,36292056,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,NaN,NaN,<NA>,6280
2316,3629,2057,36292057,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,NaN,NaN,<NA>,6360
2317,3629,2058,36292058,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,NaN,NaN,<NA>,6440
2318,3629,2059,36292059,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,NaN,NaN,<NA>,6520


## Vintage Specification

In [44]:
df_external_year_update["Vintage"] = np.select(
    [
        df_external_year_update["Year"]
        <= df_external_year_update.dropna(subset=["AADT_Historic"])["Year"].max(),
        df_external_year_update["Year"]
        >= df_external_year_update.dropna(subset=["AADT_Forecast"])["Year"].min(),
    ],
    ["Historic", "Forecast"],
    default="Current",
)

df_external_year_update


,WF_Ext,Year,;Idx_WF,segid,route,milepost,Ext_Name,AWDT_FAC,AADT_Historic,PctTrk_SU,PctTrk_MU,AADT_Forecast,AADT,Vintage
0,3601,1981,36011981,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic
1,3601,1982,36011982,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic
2,3601,1983,36011983,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic
3,3601,1984,36011984,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic
4,3601,1985,36011985,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2315,3629,2056,36292056,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,NaN,NaN,<NA>,6280,Forecast
2316,3629,2057,36292057,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,NaN,NaN,<NA>,6360,Forecast
2317,3629,2058,36292058,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,NaN,NaN,<NA>,6440,Forecast
2318,3629,2059,36292059,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,NaN,NaN,<NA>,6520,Forecast


## Passenger, Single Unit Truck, and Combo Unit Truck Volumes

In [45]:
df_external_year_update = df_external_year_update.assign(
    # Calculate and cast the individual truck volumes to nullable integer
    TRUCK_SU=lambda df: (df["AADT"] * df["PctTrk_SU"]).round().astype("Int64"),
    TRUCK_MU=lambda df: (df["AADT"] * df["PctTrk_MU"]).round().astype("Int64"),
)

df_external_year_update


,WF_Ext,Year,;Idx_WF,segid,route,milepost,Ext_Name,AWDT_FAC,AADT_Historic,PctTrk_SU,PctTrk_MU,AADT_Forecast,AADT,Vintage,TRUCK_SU,TRUCK_MU
0,3601,1981,36011981,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic,0,0
1,3601,1982,36011982,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic,0,0
2,3601,1983,36011983,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic,0,0
3,3601,1984,36011984,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic,0,0
4,3601,1985,36011985,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2315,3629,2056,36292056,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,NaN,NaN,<NA>,6280,Forecast,<NA>,<NA>
2316,3629,2057,36292057,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,NaN,NaN,<NA>,6360,Forecast,<NA>,<NA>
2317,3629,2058,36292058,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,NaN,NaN,<NA>,6440,Forecast,<NA>,<NA>
2318,3629,2059,36292059,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,NaN,NaN,<NA>,6520,Forecast,<NA>,<NA>


# Fill Missing Data

## Missing AADT and Truck Percentages

In [46]:
# Fill NaN values in AADT by aligning on WF_Ext and Year
df_external_year_update["AADT"] = (
    df_external_year_update.set_index(["WF_Ext", "Year"])["AADT"]
    .fillna(df_external_year.set_index(["WF_Ext", "Year"])["AADT"])
    .values
)

# Fill NaN values in PctTrk_SU by aligning on WF_Ext and Year
df_external_year_update["PctTrk_SU"] = (
    df_external_year_update.set_index(["WF_Ext", "Year"])["PctTrk_SU"]
    .fillna(df_external_year.set_index(["WF_Ext", "Year"])["PctTrk_SU"])
    .values
)

# Fill NaN values in PctTrk_MU by aligning on WF_Ext and Year
df_external_year_update["PctTrk_MU"] = (
    df_external_year_update.set_index(["WF_Ext", "Year"])["PctTrk_MU"]
    .fillna(df_external_year.set_index(["WF_Ext", "Year"])["PctTrk_MU"])
    .values
)

df_external_year_update


,WF_Ext,Year,;Idx_WF,segid,route,milepost,Ext_Name,AWDT_FAC,AADT_Historic,PctTrk_SU,PctTrk_MU,AADT_Forecast,AADT,Vintage,TRUCK_SU,TRUCK_MU
0,3601,1981,36011981,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.000,0.000,<NA>,0,Historic,0,0
1,3601,1982,36011982,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.000,0.000,<NA>,0,Historic,0,0
2,3601,1983,36011983,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.000,0.000,<NA>,0,Historic,0,0
3,3601,1984,36011984,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.000,0.000,<NA>,0,Historic,0,0
4,3601,1985,36011985,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.000,0.000,<NA>,0,Historic,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2315,3629,2056,36292056,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,0.141,<NA>,6280,Forecast,<NA>,<NA>
2316,3629,2057,36292057,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,0.141,<NA>,6360,Forecast,<NA>,<NA>
2317,3629,2058,36292058,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,0.141,<NA>,6440,Forecast,<NA>,<NA>
2318,3629,2059,36292059,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,0.141,<NA>,6520,Forecast,<NA>,<NA>


## Extrapolated Truck Volumes

In [47]:
# Apply extrapolation to truck counts based on AADT for each WF_Ext
df_external_year_update[["TRUCK_SU", "TRUCK_MU"]] = (
    df_external_year_update.groupby("WF_Ext", group_keys=False)
    .apply(
        lambda group: pd.DataFrame(
            {
                "TRUCK_SU": extrapolate(
                    group["TRUCK_SU"].astype(float), group["AADT"].values
                ),
                "TRUCK_MU": extrapolate(
                    group["TRUCK_MU"].astype(float), group["AADT"].values
                ),
            },
            index=group.index,
        ),
        include_groups=False,
    )
    .round()
    .astype("Int64")
)

df_external_year_update


,WF_Ext,Year,;Idx_WF,segid,route,milepost,Ext_Name,AWDT_FAC,AADT_Historic,PctTrk_SU,PctTrk_MU,AADT_Forecast,AADT,Vintage,TRUCK_SU,TRUCK_MU
0,3601,1981,36011981,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.000,0.000,<NA>,0,Historic,0,0
1,3601,1982,36011982,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.000,0.000,<NA>,0,Historic,0,0
2,3601,1983,36011983,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.000,0.000,<NA>,0,Historic,0,0
3,3601,1984,36011984,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.000,0.000,<NA>,0,Historic,0,0
4,3601,1985,36011985,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.000,0.000,<NA>,0,Historic,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2315,3629,2056,36292056,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,0.141,<NA>,6280,Forecast,0,0
2316,3629,2057,36292057,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,0.141,<NA>,6360,Forecast,0,0
2317,3629,2058,36292058,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,0.141,<NA>,6440,Forecast,0,0
2318,3629,2059,36292059,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,0.141,<NA>,6520,Forecast,0,0


## Recalculate Missing Truck Percentages

In [48]:
# Recalculate percentages only where they are currently NaN
df_external_year_update["PctTrk_SU"] = df_external_year_update["PctTrk_SU"].fillna(
    df_external_year_update["TRUCK_SU"] / df_external_year_update["AADT"]
)
df_external_year_update["PctTrk_MU"] = df_external_year_update["PctTrk_MU"].fillna(
    df_external_year_update["TRUCK_MU"] / df_external_year_update["AADT"]
)

df_external_year_update


,WF_Ext,Year,;Idx_WF,segid,route,milepost,Ext_Name,AWDT_FAC,AADT_Historic,PctTrk_SU,PctTrk_MU,AADT_Forecast,AADT,Vintage,TRUCK_SU,TRUCK_MU
0,3601,1981,36011981,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic,0,0
1,3601,1982,36011982,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic,0,0
2,3601,1983,36011983,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic,0,0
3,3601,1984,36011984,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic,0,0
4,3601,1985,36011985,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2315,3629,2056,36292056,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,0.141,<NA>,6280,Forecast,0,0
2316,3629,2057,36292057,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,0.141,<NA>,6360,Forecast,0,0
2317,3629,2058,36292058,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,0.141,<NA>,6440,Forecast,0,0
2318,3629,2059,36292059,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,0.141,<NA>,6520,Forecast,0,0


## Calculate PASSENGER counts

In [49]:
df_external_year_update = df_external_year_update.assign(
    # Calculate the passenger volumes
    PASSENGER=lambda df: (df["AADT"] - (df["TRUCK_SU"] + df["TRUCK_MU"]))
    .round()
    .astype("Int64")
)

df_external_year_update


,WF_Ext,Year,;Idx_WF,segid,route,milepost,Ext_Name,AWDT_FAC,AADT_Historic,PctTrk_SU,PctTrk_MU,AADT_Forecast,AADT,Vintage,TRUCK_SU,TRUCK_MU,PASSENGER
0,3601,1981,36011981,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic,0,0,0
1,3601,1982,36011982,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic,0,0,0
2,3601,1983,36011983,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic,0,0,0
3,3601,1984,36011984,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic,0,0,0
4,3601,1985,36011985,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2315,3629,2056,36292056,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,0.141,<NA>,6280,Forecast,0,0,6280
2316,3629,2057,36292057,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,0.141,<NA>,6360,Forecast,0,0,6360
2317,3629,2058,36292058,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,0.141,<NA>,6440,Forecast,0,0,6440
2318,3629,2059,36292059,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,0.141,<NA>,6520,Forecast,0,0,6520


## Missing AWDT_FAC from Previous Base Year Data

In [50]:
# Replace zero values with NaN in AWDT_FAC column
df_external_year_update["AWDT_FAC"] = df_external_year_update["AWDT_FAC"].replace(
    0, pd.NA
)

# Fill NaN values in AWDT_FAC by aligning on WF_Ext and Year
df_external_year_update["AWDT_FAC"] = (
    df_external_year_update.set_index(["WF_Ext", "Year"])["AWDT_FAC"]
    .fillna(df_external_year.set_index(["WF_Ext", "Year"])["AWDT_FAC"])
    .values
)

df_external_year_update


,WF_Ext,Year,;Idx_WF,segid,route,milepost,Ext_Name,AWDT_FAC,AADT_Historic,PctTrk_SU,PctTrk_MU,AADT_Forecast,AADT,Vintage,TRUCK_SU,TRUCK_MU,PASSENGER
0,3601,1981,36011981,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic,0,0,0
1,3601,1982,36011982,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic,0,0,0
2,3601,1983,36011983,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic,0,0,0
3,3601,1984,36011984,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic,0,0,0
4,3601,1985,36011985,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,0.0,<NA>,0,Historic,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2315,3629,2056,36292056,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,0.141,<NA>,6280,Forecast,0,0,6280
2316,3629,2057,36292057,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,0.141,<NA>,6360,Forecast,0,0,6360
2317,3629,2058,36292058,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,0.141,<NA>,6440,Forecast,0,0,6440
2318,3629,2059,36292059,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,0.141,<NA>,6520,Forecast,0,0,6520


# Convert to Average Weekly Number

In [51]:
df_external_year_update = df_external_year_update.assign(
    # 1. Calculate and cast the individual truck volumes to nullable integer
    AWDT=lambda df: (df["AADT"] * df["AWDT_FAC"]).round().astype("Int64"),
    PASS_VOL=lambda df: (df["PASSENGER"] * df["AWDT_FAC"]).round().astype("Int64"),
    TRUCK_MD=lambda df: (df["TRUCK_SU"] * df["AWDT_FAC"]).round().astype("Int64"),
    TRUCK_HV=lambda df: (df["TRUCK_MU"] * df["AWDT_FAC"]).round().astype("Int64"),
)

df_external_year_update


,WF_Ext,Year,;Idx_WF,segid,route,milepost,Ext_Name,AWDT_FAC,AADT_Historic,PctTrk_SU,...,AADT_Forecast,AADT,Vintage,TRUCK_SU,TRUCK_MU,PASSENGER,AWDT,PASS_VOL,TRUCK_MD,TRUCK_HV
0,3601,1981,36011981,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,...,<NA>,0,Historic,0,0,0,<NA>,<NA>,<NA>,<NA>
1,3601,1982,36011982,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,...,<NA>,0,Historic,0,0,0,<NA>,<NA>,<NA>,<NA>
2,3601,1983,36011983,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,...,<NA>,0,Historic,0,0,0,<NA>,<NA>,<NA>,<NA>
3,3601,1984,36011984,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,...,<NA>,0,Historic,0,0,0,<NA>,<NA>,<NA>,<NA>
4,3601,1985,36011985,1082_000.0,1082PM,0.0,Ext # 3601 - FAR-1082 Bird Refuge,NaN,0.0,0.0,...,<NA>,0,Historic,0,0,0,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2315,3629,2056,36292056,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,...,<NA>,6280,Forecast,0,0,6280,6534,6534,0,0
2316,3629,2057,36292057,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,...,<NA>,6360,Forecast,0,0,6360,6617,6617,0,0
2317,3629,2058,36292058,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,...,<NA>,6440,Forecast,0,0,6440,6700,6700,0,0
2318,3629,2059,36292059,1826_004.9,1826PM,4.9,Ext # 3629 - FAR-1826 South Ridge Farms,1.0404,NaN,0.187,...,<NA>,6520,Forecast,0,0,6520,6783,6783,0,0


# Visualize

In [52]:
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import interact

# ==========================================
# 1. PREPARE DATA
# ==========================================

# Filter Data: Year >= 2010
df_viz = df_external_year_update[df_external_year_update["Year"] >= 2010].copy()

# ==========================================
# 2. DEFINE PLOTTING FUNCTION
# ==========================================


def plot_station(station_name):
    # Filter for the selected station
    df_s = df_viz[df_viz["Ext_Name"] == station_name].sort_values("Year")

    if df_s.empty:
        print("No data found for this station.")
        return

    # --- Strict Vintage Slicing ---
    # Rely purely on the 'Vintage' column labels
    df_hist = df_s[df_s["Vintage"] == "Historic"]
    df_curr = df_s[df_s["Vintage"] == "Current"]
    df_fore = df_s[df_s["Vintage"] == "Forecast"]

    # --- Connectivity Logic ---
    # Manually attach the last point of the previous segment to the start of the next
    # to ensure visual continuity without recalculating year boundaries.

    # 1. Connect Historic -> Current
    if not df_hist.empty and not df_curr.empty:
        # Prepend last Historic row to Current
        df_curr = pd.concat([df_hist.iloc[[-1]], df_curr])

    # 2. Connect Current -> Forecast
    if not df_fore.empty:
        if not df_curr.empty:
            # Prepend last Current row to Forecast
            df_fore = pd.concat([df_curr.iloc[[-1]], df_fore])
        elif not df_hist.empty:
            # Direct Jump: Historic -> Forecast (if Current is missing)
            df_fore = pd.concat([df_hist.iloc[[-1]], df_fore])

    # --- Plotting ---
    fig = go.Figure()

    # TRACE ORDER: Forecast -> Current -> Historic

    # 1. Forecast (Dotted Line, Hollow Dots)
    fig.add_trace(
        go.Scatter(
            x=df_fore["Year"],
            y=df_fore["AADT"],
            mode="lines+markers",
            name="Forecast",
            line=dict(color="#D32F2F", width=2.5, dash="dot"),
            marker=dict(color="white", size=7, line=dict(color="#D32F2F", width=2)),
            hovertemplate="<b>%{x}</b>: %{y:,} (Forecast)<extra></extra>",
        )
    )

    # 2. Current (Dotted Line, Solid Dots)
    fig.add_trace(
        go.Scatter(
            x=df_curr["Year"],
            y=df_curr["AADT"],
            mode="lines+markers",
            name="Current",
            line=dict(color="#D32F2F", width=2.5, dash="dot"),
            marker=dict(color="#D32F2F", size=7, line=dict(color="#D32F2F", width=2)),
            hovertemplate="<b>%{x}</b>: %{y:,} (Current)<extra></extra>",
        )
    )

    # 3. Historic (Solid Line, Solid Dots)
    fig.add_trace(
        go.Scatter(
            x=df_hist["Year"],
            y=df_hist["AADT"],
            mode="lines+markers",
            name="Historic",
            line=dict(color="#D32F2F", width=2.5, dash="solid"),
            marker=dict(color="#D32F2F", size=7, line=dict(color="#D32F2F", width=2)),
            hovertemplate="<b>%{x}</b>: %{y:,} (Historic)<extra></extra>",
        )
    )

    # Layout
    fig.update_layout(
        title=dict(text=f"AADT Trend: <b>{station_name}</b>", font=dict(size=18)),
        xaxis=dict(
            title="Year", dtick=5, showgrid=True, gridcolor="#eee", linecolor="#333"
        ),
        yaxis=dict(
            title="AADT",
            tickformat=",",
            showgrid=True,
            gridcolor="#eee",
            linecolor="#333",
            rangemode="tozero",  # Y-axis starts at 0
        ),
        plot_bgcolor="white",
        height=500,
        legend=dict(orientation="h", y=1.05, x=1, xanchor="right"),
        margin=dict(l=40, r=40, t=80, b=40),
    )

    fig.show()


# ==========================================
# 3. CREATE WIDGET
# ==========================================

station_options = sorted(df_viz["Ext_Name"].dropna().unique())

interact(
    plot_station,
    station_name=widgets.Dropdown(
        options=station_options,
        description="Station:",
        style={"description_width": "initial"},
        layout={"width": "500px"},
    ),
)


interactive(children=(Dropdown(description='Station:', layout=Layout(width='500px'), options=('Ext # 3601 - FA…

<function __main__.plot_station(station_name)>

# Export Final Results

In [53]:
(
    df_external_year_update[
        # 1. Filter columns using the predefined list
        [
            ";Idx_WF",
            "WF_Ext",
            "Year",
            "AWDT",
            "PASS_VOL",
            "TRUCK_MD",
            "TRUCK_HV",
            "AWDT_FAC",
            "AADT",
            "PASSENGER",
            "TRUCK_SU",
            "TRUCK_MU",
            "PctTrk_SU",
            "PctTrk_MU",
        ]
    ][
        # 2. Filter rows using the boolean mask (Year >= 2010)
        df_external_year_update["Year"] >= 2010
    ]
    # 3. Export to CSV
    .to_csv("results/external_year_vol.csv", index=False)
)
